# EMG → Multi‑Label Fingers with a Spiking Neural Network (Temporal Dependence)

This notebook reimplements your pipeline using **PyTorch + snnTorch** to model temporal structure. It keeps your IIR/EMA/SMA preprocessing, adds sequence windowing for BPTT, and trains a recurrent spiking model to predict 5 binary finger activations (multi‑label) at each timestep.

**Key changes vs. your MLP version**
- Convert continuous stream into sequences of length `T` for temporal learning.
- Use recurrent Leaky‑Integrate‑and‑Fire (R‑LIF) layers (snnTorch) with BPTT.
- Multi‑label output with sigmoid + BCE loss at *every timestep* (teacher signals aligned with delay).
- Metrics: R² (per finger and macro), binary accuracy, and AUROC (optional) computed on flattened frames.
- Same save format (params JSON) to ease downstream inference.
> Tip: If you don't want recurrence, you can switch to feed‑forward LIF across time by replacing RLeaky with Leaky.


## 0) Imports & Config


In [1]:
import os
import re
import json
import math
import time
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional

# plotting
import matplotlib.pyplot as plt

# ML
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# spikes
import snntorch as snn
from snntorch import utils as snn_utils
from snntorch import surrogate

# metrics
from sklearn.metrics import r2_score, f1_score, roc_auc_score, accuracy_score

# your preprocessing modules
from exg.ema import EMA
from exg.sma import SMA
from exg.iir import IIR

# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE)

# Repro
torch.manual_seed(42)
np.random.default_rng(42)

Device: mps


Generator(PCG64) at 0x33C164580